# Population Growth Study Book
## End-to-End Analysis & Prediction

### 1. Data Cleaning & Transformation

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

df = pd.read_csv('popolation_growth.csv')
df_clean = df.drop(columns=['1960', 'Indicator Name', 'Indicator Code'])
df_long = df_clean.melt(id_vars=['Country Name', 'Country Code'], var_name='Year', value_name='Growth').dropna()
df_long['Year'] = pd.to_numeric(df_long['Year'])
df_long = df_long.sort_values(['Country Code', 'Year'])
df_long.head()

### 2. Statistical EDA

In [ ]:
print(f'Skewness: {df_long["Growth"].skew():.4f}')
print(f'Kurtosis: {df_long["Growth"].kurt():.4f}')
sns.boxplot(x=df_long['Growth'])
plt.title('Distribution of Population Growth')

### 3. Feature Engineering

In [ ]:
df_ml = df_long.copy()
df_ml['Lag_1'] = df_ml.groupby('Country Code')['Growth'].shift(1)
df_ml['Rolling_3'] = df_ml.groupby('Country Code')['Growth'].transform(lambda x: x.shift(1).rolling(3).mean())
df_ml = df_ml.dropna()

from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df_ml['Country_ID'] = le.fit_transform(df_ml['Country Code'])

### 4. Modeling & Hyperparameter Tuning

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import RobustScaler

X = df_ml[['Year', 'Country_ID', 'Lag_1', 'Rolling_3']]
y = df_ml['Growth']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = GradientBoostingRegressor(random_state=42)
param_grid = {'n_estimators': [100, 200], 'max_depth': [3, 5]}
grid = GridSearchCV(model, param_grid, cv=3)
grid.fit(X_train_scaled, y_train)
best_model = grid.best_estimator_

### 5. Final Evaluation & Deployment

In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score
y_pred = best_model.predict(X_test_scaled)
print(f'MAE: {mean_absolute_error(y_test, y_pred):.4f}')
print(f'R2: {r2_score(y_test, y_pred):.4f}')

plt.scatter(y_test, y_pred, alpha=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.title('Actual vs Predicted')
plt.show()